In [8]:
from full_reward import *

In [13]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from contextlib import redirect_stdout

def _pad_to_max_length(curves, pad_value=np.nan):
    max_len = max(len(c) for c in curves)
    out = np.full((len(curves), max_len), pad_value, dtype=float)
    for i, c in enumerate(curves):
        out[i, :len(c)] = c
    return out

def run_5_seeds_and_save_curves(
    train_fn,
    train_kwargs=None,
    base_train_kwargs=None,  # compat: allow old name
    seeds=(0,1,2,3,4),
    out_dir="plot1_model_free",
    label="experiment",
    silence_prints=True,
):
    """
    Generic runner for any train() that returns:
      policy, queries, steps, average_returns

    Accepts kwargs via either:
      - train_kwargs (preferred)
      - base_train_kwargs (backwards compatible)

    Saves:
      - per-seed curves (.npy)
      - mean/std curve (.npy)
      - x axis = update index (.npy)
      - plot (.png)
      - summary (.json)
      - per-seed logs (optional)
    """
    if train_kwargs is None:
        train_kwargs = base_train_kwargs
    if train_kwargs is None:
        raise ValueError("Provide train_kwargs (or base_train_kwargs)")

    os.makedirs(out_dir, exist_ok=True)

    per_seed_curves = []
    final_returns = []
    total_steps = []
    total_queries = []

    for seed in seeds:
        kwargs = dict(train_kwargs)
        kwargs["seed"] = seed

        log_path = os.path.join(out_dir, f"{label}_seed{seed}_log.txt")
        print(f"\n=== Running seed {seed} ===")

        if silence_prints:
            with open(log_path, "w") as f, redirect_stdout(f):
                policy, queries, steps, average_returns = train_fn(**kwargs)
        else:
            policy, queries, steps, average_returns = train_fn(**kwargs)

        curve = np.asarray(average_returns, dtype=float)
        per_seed_curves.append(curve)

        final_returns.append(float(curve[-1]) if len(curve) else np.nan)
        total_steps.append(int(steps))
        total_queries.append(int(queries))

        np.save(os.path.join(out_dir, f"{label}_seed{seed}_curve.npy"), curve)

    curves_mat = _pad_to_max_length(per_seed_curves, pad_value=np.nan)
    mean_curve = np.nanmean(curves_mat, axis=0)
    std_curve  = np.nanstd(curves_mat, axis=0)

    x_updates = np.arange(len(mean_curve), dtype=float)

    np.save(os.path.join(out_dir, f"{label}_mean_curve.npy"), mean_curve)
    np.save(os.path.join(out_dir, f"{label}_std_curve.npy"), std_curve)
    np.save(os.path.join(out_dir, f"{label}_x_updates.npy"), x_updates)

    summary = {
        "label": label,
        "seeds": list(seeds),
        "train_kwargs": train_kwargs,
        "final_return_per_seed": final_returns,
        "final_return_mean": float(np.nanmean(final_returns)),
        "final_return_std": float(np.nanstd(final_returns)),
        "total_steps_per_seed": total_steps,
        "total_queries_per_seed": total_queries,
        "curve_len_per_seed": [int(len(c)) for c in per_seed_curves],
    }
    with open(os.path.join(out_dir, f"{label}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # Plot
    plt.figure(figsize=(8, 4.8))
    for c in per_seed_curves:
        plt.plot(np.arange(len(c)), c, alpha=0.25, linewidth=1)

    plt.plot(x_updates, mean_curve, linewidth=2, label="mean (5 seeds)")
    plt.fill_between(
        x_updates,
        mean_curve - std_curve,
        mean_curve + std_curve,
        alpha=0.2,
        label="±1 std",
    )

    plt.xlabel("Training update step (index into average_returns)")
    plt.ylabel("Avg true return")
    plt.title(label)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    fig_path = os.path.join(out_dir, f"{label}.png")
    plt.savefig(fig_path, dpi=200)
    plt.close()

    print("\nSaved:", fig_path)
    print("Final avg true return over seeds:",
          f"{summary['final_return_mean']:.4f} ± {summary['final_return_std']:.4f}")

    return summary


In [14]:
noise_list = [0.1, 0.5, 0.8]

base_kwargs = dict(
    m=50,
    eta=0.5,
    epsilon=0.1,
    grid_size=8,
    danger=[7,1],
    goal=[4,5],
    wall=[2,5],
    horizon=50,
    coins=None,
)

for sparse in [False, True]:
    for noise in noise_list:
        kwargs = dict(base_kwargs)
        kwargs["noise"] = noise
        kwargs["sparse"] = sparse

        label = f"full_reward_noise{noise}_sparse{sparse}"
        run_5_seeds_and_save_curves(
            train_fn=train,
            base_train_kwargs=kwargs,
            seeds=(0,1,2,3,4),
            out_dir="plot1_model_free_full_reward",
            label=label,
            silence_prints=True,
        )



=== Running seed 0 ===

=== Running seed 1 ===

=== Running seed 2 ===

=== Running seed 3 ===

=== Running seed 4 ===

Saved: plot1_model_free_full_reward/full_reward_noise0.1_sparseFalse.png
Final avg true return over seeds: 30.0851 ± 2.5429

=== Running seed 0 ===

=== Running seed 1 ===

=== Running seed 2 ===

=== Running seed 3 ===

=== Running seed 4 ===

Saved: plot1_model_free_full_reward/full_reward_noise0.5_sparseFalse.png
Final avg true return over seeds: 31.1154 ± 0.0785

=== Running seed 0 ===

=== Running seed 1 ===

=== Running seed 2 ===

=== Running seed 3 ===

=== Running seed 4 ===

Saved: plot1_model_free_full_reward/full_reward_noise0.8_sparseFalse.png
Final avg true return over seeds: 22.9999 ± 6.4851

=== Running seed 0 ===

=== Running seed 1 ===

=== Running seed 2 ===

=== Running seed 3 ===

=== Running seed 4 ===

Saved: plot1_model_free_full_reward/full_reward_noise0.1_sparseTrue.png
Final avg true return over seeds: 12.6330 ± 0.0899

=== Running seed 0 =

In [15]:
thresholds_by_sparse = {
    False: [16, 28],
    True:  [6, 12],
}
for sparse, thresh_list in thresholds_by_sparse.items():
    print(sparse,thresh_list)

False [16, 28]
True [6, 12]
